# Setup

In [1]:
master = "spark://spark-master:7077"
dataFolder = "/data/" if master.startswith("spark://") else "../data/"

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.sql import Window

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-profiling-exercises")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

# if master.startswith("spark://"):
#     spark_builder = (spark_builder
#                       .config("spark.driver.host", "localhost")
#                       .config("spark.driver.bindAddress", "0.0.0.0"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 09:30:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://dc5139ca3cdd:4040
Default parallelism (cores seen by the driver): 6


In [3]:
import sys
sys.path.append("/workspace/apps")

from spark_profiler import SparkProfiler
profiler = SparkProfiler(spark)



In [4]:
months = [1,2,3,4,5]
files = [dataFolder + f"yellow_tripdata_2020-{m:02d}.parquet" for m in months]

trips = spark.read.parquet(*files)
zones = spark.read.csv(dataFolder + "taxi_zone_lookup.csv", header=True, inferSchema=True)

## Integração do M1
### Data ingestion and cleaning
Load the monthly Parquet files. Enforce the schema
(cast types, handle nulls)

In [5]:
from pyspark.sql import functions as F
with (((profiler.profile("1-ingestion-cleaning")))):

    trips_typed = (trips
        .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("trip_distance",   F.col("trip_distance").cast("double"))
        .withColumn("fare_amount",     F.col("fare_amount").cast("double"))
        .withColumn("PULocationID",    F.col("PULocationID").cast("int"))
        .withColumn("DOLocationID",    F.col("DOLocationID").cast("int"))
        .withColumn("year", F.year("tpep_pickup_datetime"))
    )
    trips_typed = (trips_typed
                    .filter("fare_amount > 0")
                    .filter("trip_distance > 0")
                    .filter("passenger_count > 0 ")
                    .filter("passenger_count < 6")
                    .filter("tpep_dropoff_datetime > tpep_pickup_datetime")
                    .filter("PULocationID BETWEEN 1 AND 263")
                    .filter("DOLocationID BETWEEN 1 AND 263"))
    print(trips.count())
    print(trips_typed.count())


16298550
15070973
[1-ingestion-cleaning] wall time: 5.79s (6 stages, 26 tasks) - see profiler.report() / profiler.compare(...)


# M2

In [6]:
with profiler.profile("2-temporal-demand-analysis"):
    trips2 = (trips_typed
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60))

    joined = (trips2
        .join(zones, trips2.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    m2 = (joined
        .groupBy("pickup_borough", "day_of_week", "hour_of_day")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare"),
            F.avg("duration_min").alias("avg_duration_min"),
            F.avg("trip_distance").alias("avg_distance"),
        ))

    m2.count()

    m2.orderBy("pickup_borough", "day_of_week", "hour_of_day").show(20)

    (m2.write
        .mode("overwrite")
        .partitionBy("pickup_borough")
        .parquet("/data/output/m2_temporal_demand"))



+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|          avg_fare|  avg_duration_min|      avg_distance|
+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|         Bronx|          1|          0|        91| 15.82967032967033|28.519963369963374| 4.500329670329671|
|         Bronx|          1|          1|        84|15.011904761904763|13.252380952380955|3.8738095238095234|
|         Bronx|          1|          2|        65| 16.98723076923077| 33.08179487179487| 4.934769230769231|
|         Bronx|          1|          3|        79|14.753164556962025|12.410337552742616|4.0458227848101265|
|         Bronx|          1|          4|        83|13.975903614457831|10.375903614457831| 3.863012048192772|
|         Bronx|          1|          5|        63|16.547619047619047| 11.55026455026455|  4.75095238095238|
|         Bronx|   

# Integração M3
## Zone-level statistics
For each pickup zone and month, compute mean and
standard deviation of fare_amount and trip_distance


In [7]:
trips_typed.printSchema()
zones.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- year: integer (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Z

In [8]:
with (profiler.profile("M3- Zone Level statistics")):
    mean_and_sdev = trips_typed.groupBy("PULocationID")

    mean_and_sdev = (trips_typed
                     .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
        .groupBy("PULocationID", "month")
        .agg(
            F.count("*").alias("trips"),
            F.avg("fare_amount").alias("fare_mean"),
            F.stddev("fare_amount").alias("fare_std"),
            F.avg("trip_distance").alias("dist_mean"),
            F.stddev("trip_distance").alias("dist_std"),
        ))
    # mean_and_sdev.printSchema()
    # nomes_final = mean_and_sdev.join(zones, mean_and_sdev["PULocationID"] == zones["LocationID"])
    # nomes_final.select("Zone", "Month", "fare_mean", "fare_std", "dist_mean", "dist_std").show()


[M3- Zone Level statistics] wall time: 0.07s (0 stages, 0 tasks) - see profiler.report() / profiler.compare(...)


# O1 — Fare anomaly detection.
## For each (PULocationID, hour_of_day) bucket,

compute mean and σ of fare_amount / trip_distance .
flag trips where the per-km fare exceeds mean + 3σ.
compare weekday vs. weekend anomaly rates broken down by payment_type and pickup
borough.

(Mais rapido)

In [9]:
from pyspark.sql import Window

w = Window.partitionBy("PULocationID", "hour_of_day")

with profiler.profile("O1-fare-anomaly-window"):
    flagged = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("fpm_mean", F.avg("fare_per_mile").over(w))
        .withColumn("fpm_std",  F.stddev("fare_per_mile").over(w))
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)


[O1-fare-anomaly-window] wall time: 4.41s (7 stages, 22 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 

(Mais lento)

In [10]:
with profiler.profile("O1-fare-anomaly"):
    o1_base = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std")))

    flagged = flagged.withColumn("is_weekend", F.col("day_of_week").isin(1, 7))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.compare("O1-fare-anomaly-window", "O1-fare-anomaly")

[O1-fare-anomaly] wall time: 8.35s (14 stages, 59 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.0059

,O1-fare-anomaly-window,O1-fare-anomaly,change
metric,,,
wall_time_s,4.41,8.35,+89.3%
num_jobs,4,7,+75.0%
num_stages,7,14,+100.0%
num_tasks,22,59,+168.2%
executor_run_time_ms,10423,30714,+194.7%
executor_cpu_time_ms,9857,29377,+198.0%
input_bytes,753.0KB,1.5MB,+98.4%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,11.3KB,33.5MB,+303353.8%


Versão com persist

In [11]:
from pyspark import StorageLevel

with profiler.profile("O1-fare-anomaly-persist"):
    o1_base = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .persist(StorageLevel.MEMORY_AND_DISK))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1_base.unpersist()
o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.compare("O1-fare-anomaly-persist", "O1-fare-anomaly")

[O1-fare-anomaly-persist] wall time: 12.33s (16 stages, 71 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  2573

,O1-fare-anomaly-persist,O1-fare-anomaly,change
metric,,,
wall_time_s,12.33,8.35,-32.3%
num_jobs,9,7,-22.2%
num_stages,16,14,-12.5%
num_tasks,71,59,-16.9%
executor_run_time_ms,52875,30714,-41.9%
executor_cpu_time_ms,44231,29377,-33.6%
input_bytes,642.4MB,1.5MB,-99.8%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,33.5MB,33.5MB,+0.0%


Não valeu a pena, quase acabou com o meu pc e nem foi mais rapido

### Versão sem o Broadcast join, para comparar no relatorio

In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)

with profiler.profile("O1-no-broadcast"):
    o1_base = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.explain()

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.adaptive.autoBroadcastJoinThreshold")

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)

profiler.compare("O1-fare-anomaly", "O1-no-broadcast")

[O1-no-broadcast] wall time: 9.74s (24 stages, 112 tasks) - see profiler.report() / profiler.compare(...)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_borough#1433, payment_type#9L, is_weekend#1432], functions=[count(1), avg(cast(is_anomaly#1431 as int))])
   +- Exchange hashpartitioning(pickup_borough#1433, payment_type#9L, is_weekend#1432, 200), ENSURE_REQUIREMENTS, [plan_id=3915]
      +- HashAggregate(keys=[pickup_borough#1433, payment_type#9L, is_weekend#1432], functions=[partial_count(1), partial_avg(cast(is_anomaly#1431 as int))])
         +- Project [payment_type#9L, is_anomaly#1431, is_weekend#1432, Borough#37 AS pickup_borough#1433]
            +- SortMergeJoin [PULocationID#46], [LocationID#36], LeftOuter
               :- Sort [PULocationID#46 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(PULocationID#46, 200), ENSURE_REQUIREMENTS, [plan_id=3907]
               :     +- Project [PULocationID#46, payment_ty

,O1-fare-anomaly,O1-no-broadcast,change
metric,,,
wall_time_s,8.35,9.74,+16.6%
num_jobs,7,7,+0.0%
num_stages,14,24,+71.4%
num_tasks,59,112,+89.8%
executor_run_time_ms,30714,39200,+27.6%
executor_cpu_time_ms,29377,38030,+29.5%
input_bytes,1.5MB,1.5MB,+0.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,33.5MB,59.9MB,+78.7%


### Verificar com varias partições

In [13]:
def run_o1(df):
    o1_base = (df
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))
    stats = (o1_base.groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))
    flagged = (o1_base.join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))
    flagged = (flagged.join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))
    return (flagged.groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

spark.conf.set("spark.sql.adaptive.enabled", "false")

for n in [6, 12, 50, 200, 1000]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    with profiler.profile(f"O1-partitions-{n}"):
        run_o1(trips_typed).count()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", 200)

profiler.report()

[O1-partitions-6] wall time: 9.28s (6 stages, 26 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-12] wall time: 7.95s (6 stages, 38 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-50] wall time: 7.73s (6 stages, 114 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-200] wall time: 8.69s (6 stages, 414 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-1000] wall time: 13.11s (6 stages, 2014 tasks) - see profiler.report() / profiler.compare(...)


,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
1-ingestion-cleaning,5.79,4,6,26,11937,9273,841.9KB,0.0B,696.0B,696.0B,0.0B,0.0B
2-temporal-demand-analysis,15.86,10,15,50,35844,33629,2.2MB,36.1KB,563.2KB,563.2KB,0.0B,0.0B
M3- Zone Level statistics,0.07,0,0,0,0,0,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
O1-fare-anomaly-window,4.41,4,7,22,10423,9857,753.0KB,0.0B,11.3KB,11.3KB,0.0B,0.0B
O1-fare-anomaly,8.35,7,14,59,30714,29377,1.5MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-fare-anomaly-persist,12.33,9,16,71,52875,44231,642.4MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-no-broadcast,9.74,7,24,112,39200,38030,1.5MB,0.0B,59.9MB,59.9MB,128.0MB,5.0MB
O1-partitions-6,9.28,2,6,26,33263,32488,1.5MB,0.0B,69.9MB,69.9MB,0.0B,0.0B
O1-partitions-12,7.95,2,6,38,29928,29222,1.5MB,0.0B,62.9MB,62.9MB,0.0B,0.0B


### Select

In [14]:
with profiler.profile("O1-select-cols"):
    o1_base = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones.select("LocationID", "Borough"),
              flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)

[O1-select-cols] wall time: 6.90s (14 stages, 59 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.00598

### Window com o select, mistura

In [15]:
from pyspark.sql import Window

w = Window.partitionBy("PULocationID", "hour_of_day")

with profiler.profile("O1-window-select"):
    flagged = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1, 7))
        .select("PULocationID", "payment_type", "hour_of_day", "is_weekend", "fare_per_mile")
        .withColumn("fpm_mean", F.avg("fare_per_mile").over(w))
        .withColumn("fpm_std",  F.stddev("fare_per_mile").over(w))
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std")))

    flagged = (flagged
        .join(F.broadcast(zones.select("LocationID", "Borough")),
              flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.report()

[O1-window-select] wall time: 3.85s (7 stages, 22 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.0059

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
1-ingestion-cleaning,5.79,4,6,26,11937,9273,841.9KB,0.0B,696.0B,696.0B,0.0B,0.0B
2-temporal-demand-analysis,15.86,10,15,50,35844,33629,2.2MB,36.1KB,563.2KB,563.2KB,0.0B,0.0B
M3- Zone Level statistics,0.07,0,0,0,0,0,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
O1-fare-anomaly-window,4.41,4,7,22,10423,9857,753.0KB,0.0B,11.3KB,11.3KB,0.0B,0.0B
O1-fare-anomaly,8.35,7,14,59,30714,29377,1.5MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-fare-anomaly-persist,12.33,9,16,71,52875,44231,642.4MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-no-broadcast,9.74,7,24,112,39200,38030,1.5MB,0.0B,59.9MB,59.9MB,128.0MB,5.0MB
O1-partitions-6,9.28,2,6,26,33263,32488,1.5MB,0.0B,69.9MB,69.9MB,0.0B,0.0B
O1-partitions-12,7.95,2,6,38,29928,29222,1.5MB,0.0B,62.9MB,62.9MB,0.0B,0.0B


# O2 — Origin-destination flow analysis.
Compute monthly trip counts for all (PULocationID, DOLocationID) pairs. Join the zone
lookup to resolve zone names and boroughs.
Compute each route's share of total outbound trips from its pickup zone per month.
Identify the top-20 routes per pickup borough and detect routes whose monthly share reverses
direction between Q1 and Q3.

In [16]:
from pyspark.sql import Window

with profiler.profile("O2-od-flow"):
    od = (trips_typed
        .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
        .groupBy("PULocationID", "DOLocationID", "month")
        .agg(F.count("*").alias("trips")))

    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"))
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Borough").alias("dropoff_borough"),
                      F.col("Zone").alias("dropoff_zone"))
    od = od.join(pu, "PULocationID", "left").join(do, "DOLocationID", "left")

    w_out = Window.partitionBy("PULocationID", "month")
    od = od.withColumn("share", F.col("trips") / F.sum("trips").over(w_out))

    w_rank = Window.partitionBy("pickup_borough", "month").orderBy(F.col("trips").desc())
    top20 = (od.withColumn("rank", F.row_number().over(w_rank))
               .filter(F.col("rank") <= 20))

    top20.count()

top20.orderBy("pickup_borough", "month", "rank").show(40, truncate=False)

[O2-od-flow] wall time: 4.17s (12 stages, 32 tasks) - see profiler.report() / profiler.compare(...)
+------------+------------+-------------------+-----+--------------+---------------------------------+---------------+---------------------------------+--------------------+----+
|DOLocationID|PULocationID|month              |trips|pickup_borough|pickup_zone                      |dropoff_borough|dropoff_zone                     |share               |rank|
+------------+------------+-------------------+-----+--------------+---------------------------------+---------------+---------------------------------+--------------------+----+
|78          |78          |2009-01-01 00:00:00|1    |Bronx         |East Tremont                     |Bronx          |East Tremont                     |1.0                 |1   |
|168         |168         |2020-01-01 00:00:00|194  |Bronx         |Mott Haven/Port Morris           |Bronx          |Mott Haven/Port Morris           |0.24556962025316456 |1   |
|74  